# **Assignment 2: Problem 2 - Character-Level Name Generation Using RNN Variants**

*   Name: Shrusti Vipulbhai Jain
*   Roll No: M25CSE030

In [ ]:
# Importing necessary files

import re
import random
from difflib import SequenceMatcher

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Checking device used for computation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Reading names from a text file

def load_names(file_path):
    """
    In this function, we read file line by line.
    Also, removed empty lines and extra spaces.
    This returns clean list of names.
    """

    with open(file_path, 'r', encoding='utf-8') as f:
        names = f.readlines()

    # Remove newline characters and strip spaces
    names = [name.strip() for name in names if name.strip() != ""]

    return names

In [ ]:
# Reading names from file and displaying it

file_path = "TrainingNames.txt"
raw_names = load_names(file_path)

print("Total names loaded:", len(raw_names))
print("Sample names:", raw_names[:10])

Total names loaded: 1000
Sample names: ['Arnaar', 'Aryaan', 'Bhsan', 'Yavii', 'Ankaa', 'Suyaansh', 'Dehai', 'Lamaya', 'Lataaya', 'Dashaesh']


In [ ]:
# Cleaning of single name by removing unwanted characters

def clean_name(name):
    """
    In this function, we clean single names.
    Here, lowercase conversion of names done.
    Further, remove non-alphabet characters if present.
    """

    # Convert to lowercase
    name = name.lower()

    # Keep only alphabets (remove numbers, symbols)
    name = re.sub(r'[^a-z]', '', name)

    return name

In [ ]:
# Apply cleaning to all names and removeal of invalid ones

def preprocess_names(names):
    """
    In this function, we perform cleaning of all names.
    Also, remove very short names to make training data less noisy.
    """

    cleaned_names = []

    for name in names:
        clean = clean_name(name)

        # Keep only meaningful names having length greater than 1
        if len(clean) > 1:
            cleaned_names.append(clean)

    return cleaned_names

In [ ]:
# Performing cleaning and preprocessing on names
names = preprocess_names(raw_names)

print("After cleaning:", len(names))
print("Sample cleaned names:", names[:10])

After cleaning: 1000
Sample cleaned names: ['arnaar', 'aryaan', 'bhsan', 'yavii', 'ankaa', 'suyaansh', 'dehai', 'lamaya', 'lataaya', 'dashaesh']


In [ ]:
# Adding <SOS> and <EOS> tokens to each name which mark start and end of names

def add_tokens(names):
    """
    In this function, we add <SOS> and <EOS> tokens to each name.
    This are special start and end tokens.
    Help model to understand where sequence or name begins and ends during training and generation.
    """

    tokenized_names = []

    for name in names:
        # Add tokens as characters
        new_name = ['<SOS>'] + list(name) + ['<EOS>']
        tokenized_names.append(new_name)

    return tokenized_names

In [ ]:
# Tokenizing names and displaying sample tokenized name

tokenized_names = add_tokens(names)

print("Example tokenized name:", tokenized_names[0])

Example tokenized name: ['<SOS>', 'a', 'r', 'n', 'a', 'a', 'r', '<EOS>']


In [ ]:
# Building vocabulary by character-to-index and index-to-character mappings

def build_vocab(tokenized_names):
    """
    In this function, we collects all unique characters to make vocabulary.
    Further, create mappings for character to number and number to character.
    This is done because numerical vectors required for model training.
    """

    vocab = set()

    for name in tokenized_names:
        for char in name:
            vocab.add(char)

    vocab = sorted(list(vocab))

    char_to_index = {char: idx for idx, char in enumerate(vocab)}
    index_to_char = {idx: char for char, idx in char_to_index.items()}

    return vocab, char_to_index, index_to_char

In [ ]:
# Making vocabulary, then checking vocabulary and it's size

vocab, char_to_index, index_to_char = build_vocab(tokenized_names)

print("Vocabulary size:", len(vocab))
print("Vocabulary:", vocab)

Vocabulary size: 26
Vocabulary: ['<EOS>', '<SOS>', 'a', 'b', 'c', 'd', 'e', 'f', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z']


In [ ]:
# Converting tokenized names into numerical format

def encode_names(tokenized_names, char_to_index):
    """
    In this function, we convert each tokenized name into a sequence of integers.
    Done using vocabulary mapping of character to number.
    This numerical format is required for training models.
    Because they cannot handle character data directly.
    """

    encoded_data = []

    for name in tokenized_names:
        encoded = [char_to_index[char] for char in name]
        encoded_data.append(encoded)

    return encoded_data

In [ ]:
# Encoding tokenized names to numerical format

encoded_names = encode_names(tokenized_names, char_to_index)

print("Example encoded name:", encoded_names[0])

Example encoded name: [1, 2, 18, 14, 2, 2, 18, 0]


In [ ]:
class NameDataset(Dataset):
    """
    In this dataset class, we prepare input and target sequences for training.
    Here, input contains all characters except last one.
    While, target is shifted by one position.
    Overall, this helps model learn next-character prediction.
    So, it will finally generate new names.
    """

    def __init__(self, encoded_names):
        self.data = encoded_names

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data[idx]

        # Input and target split
        input_seq = sequence[:-1]
        target_seq = sequence[1:]

        return torch.tensor(input_seq), torch.tensor(target_seq)

In [ ]:
# Padding sequences in batch to same length.

def pad_sequences(batch):
    """
    In this function, we pads all sequences.
    Because names have varying lengths.
    So, shorter sequences are padded with zeros to match them to longest sequence in batch.
    Done to ensures efficient batch processing.
    """

    inputs, targets = zip(*batch)

    input_lengths = [len(seq) for seq in inputs]
    max_len = max(input_lengths)

    padded_inputs = []
    padded_targets = []

    for inp, tgt in zip(inputs, targets):
        pad_length = max_len - len(inp)

        padded_inputs.append(
            torch.cat([inp, torch.zeros(pad_length, dtype=torch.long)])
        )

        padded_targets.append(
            torch.cat([tgt, torch.zeros(pad_length, dtype=torch.long)])
        )

    return torch.stack(padded_inputs), torch.stack(padded_targets)

In [ ]:
# Creating dataset
dataset = NameDataset(encoded_names)

# Creating dataloader
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_sequences
)

# Checking one batch
for inputs, targets in dataloader:
    print("Input shape:", inputs.shape)
    print("Target shape:", targets.shape)
    break

Input shape: torch.Size([32, 9])
Target shape: torch.Size([32, 9])


## Task 1: Model Implementation

### A. Vanilla Recurrent Neural Network (RNN)

In [ ]:
class VanillaRNN(nn.Module):
    """
    In this class, we implement basic RNN model.
    Here, we define embedding layer to convert input indices into dense vectors.
    Then, two linear layers are used to process input and previous hidden state.
    Further, dropout layer is added to reduce overfitting.
    This has output layer which then at the end maps hidden state to vocabulary predictions.
    """

    def __init__(self, vocab_size, embedding_dim, hidden_size, dropout=0.3):
        super(VanillaRNN, self).__init__()

        # Save hyperparameters
        self.hidden_size = hidden_size

        # Embedding layer (converts indices → dense vectors)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN weights (manual implementation)
        self.Wxh = nn.Linear(embedding_dim, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size)

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Output layer (hidden → vocab)
        self.output_layer = nn.Linear(hidden_size, vocab_size)

        # Activation
        self.activation = torch.tanh

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        batch_size, seq_len = x.size()

        # Initialize hidden state
        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        outputs = []

        # Convert input to embeddings
        x_embed = self.embedding(x)

        # Loop through sequence
        for t in range(seq_len):
            x_t = x_embed[:, t, :]

            # RNN equation
            h = self.activation(self.Wxh(x_t) + self.Whh(h))

            # Apply dropout
            h = self.dropout(h)

            # Output prediction
            y_t = self.output_layer(h)
            outputs.append(y_t.unsqueeze(1))

        # Concatenate all outputs
        outputs = torch.cat(outputs, dim=1)

        return outputs

### B. Bidirectional Long Short-Term Memory (BLSTM)

In [ ]:
class BLSTMModel(nn.Module):
    """
    In this class, we implement Bidirectional LSTM model.
    Here, embedding layer converts input into vectors.
    Further, LSTM processes sequence in both forward and backward directions.
    Then, dropout is used to reduce overfitting.
    Also, output layer maps hidden states to vocabulary predictions.
    """

    def __init__(self, vocab_size, embedding_dim, hidden_size, dropout=0.3):
        super(BLSTMModel, self).__init__()

        self.hidden_size = hidden_size

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Bidirectional LSTM + dropout
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        # Output layer (hidden*2 because bidirectional)
        self.output_layer = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        # Convert to embeddings
        x_embed = self.embedding(x)

        # LSTM output
        lstm_out, _ = self.lstm(x_embed)

        # Apply dropout
        lstm_out = self.dropout(lstm_out)

        # Pass through output layer
        output = self.output_layer(lstm_out)

        return output

### C. RNN with Basic Attention Mechanism

In [ ]:
class RNNAttentionModel(nn.Module):
    """
    In this class, we extend basic RNN with attention mechanism.
    Here, embedding layer converts inputs into vectors.
    Also, RNN layers compute hidden states step by step.
    Further, attention layer is added to focus on important parts of sequence.
    Then, dropout is used to improve generalization.
    """

    def __init__(self, vocab_size, embedding_dim, hidden_size, dropout=0.3):
        super(RNNAttentionModel, self).__init__()

        self.hidden_size = hidden_size

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN weights (manual)
        self.Wxh = nn.Linear(embedding_dim, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size)

        # Attention projection
        self.attn = nn.Linear(hidden_size, hidden_size)

        # Add dropout
        self.dropout = nn.Dropout(dropout)

        # Output layer
        self.output_layer = nn.Linear(hidden_size, vocab_size)

        # Activation
        self.activation = torch.tanh

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        batch_size, seq_len = x.size()

        h = torch.zeros(batch_size, self.hidden_size).to(x.device)

        x_embed = self.embedding(x)

        hidden_states = []

        # First pass
        for t in range(seq_len):
            x_t = x_embed[:, t, :]

            h = self.activation(self.Wxh(x_t) + self.Whh(h))

            h = self.dropout(h)

            hidden_states.append(h.unsqueeze(1))

        # Stack hidden states
        hidden_states = torch.cat(hidden_states, dim=1)  # (batch, seq_len, hidden)

        outputs = []

        # Second pass: apply attention
        for t in range(seq_len):
            h_t = hidden_states[:, t, :].unsqueeze(1)  # (batch, 1, hidden)

            # Project before attention (NEW)
            h_proj = self.attn(hidden_states)

            # Dot-product attention
            scores = torch.bmm(h_t, hidden_states.transpose(1, 2))  # (batch, 1, seq_len)

            attention_weights = torch.softmax(scores, dim=-1)

            # Context vector
            context = torch.bmm(attention_weights, hidden_states)  # (batch, 1, hidden)

            context = context.squeeze(1)

            context = self.dropout(context)

            # Output
            y_t = self.output_layer(context)
            outputs.append(y_t.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)

        return outputs

### D. Training Function

In [ ]:
# Initializing loss function and optimizer for training

def setup_training(model, learning_rate):
    """
    In this function, we set up components required for training.
    Here, Cross entropy loss is used for classification of next character.
    Also, Padding index is ignored so model does not learn from padded values.
    Further, Label smoothing is applied to avoid overconfident predictions.
    This uses Adam optimizer update model weights efficiently.
    """

    # Cross entropy loss for classification, while ignore padding index
    criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

    # Adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    return criterion, optimizer

In [ ]:
# Counting total trainable parameters in model

def count_parameters(model):
    """
    In this function, we count total trainable parameters in model.
    Here, only parameters requiring gradients are considered.
    So, this gives total number of weights updated during training.
    """

    total_params = 0

    for param in model.parameters():
        if param.requires_grad:
            total_params += param.numel()

    return total_params

In [ ]:
# Computes number of parameters and model size in MB.

def get_model_size(model):
    """
    In this function, we calculate the total number of trainable parameters in the model.
    Each parameter is counted using numel(), which gives total elements.
    To compute model size, we assume each parameter takes 4 bytes (float32).
    The total size is converted into megabytes for easier interpretation.
    """

    total_params = 0

    for param in model.parameters():
        if param.requires_grad:
            total_params += param.numel()

    # Each parameter = 4 bytes (float32)
    model_size_bytes = total_params * 4

    model_size_mb = model_size_bytes / (1024 ** 2)

    return total_params, model_size_mb

In [ ]:
# Function for training model on given data

def train_model(model, dataloader, criterion, optimizer, num_epochs):
    """
    In this function, we train model over multiple epochs.
    Also, for each batch inputs and targets are passed through the model.
    Further, Loss is computed between predicted and actual outputs.
    And, gradients are calculated for weights updated.
    """

    model.to(device)

    for epoch in range(num_epochs):
        total_loss = 0

        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Reshape for loss calculation
            outputs = outputs.view(-1, outputs.size(-1))
            targets = targets.view(-1)

            # Compute loss
            loss = criterion(outputs, targets)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [ ]:
# Generating name for various models

def generate_name(model, char_to_index, index_to_char, max_length=12, temperature=0.6, top_k=5, top_p=0.8):
    """
    In this function, we generate name character by character.
    Here, we starts with a start token and continues step by step.
    Then, at each step next character is sampled from predicted probabilities.
    Further, generation stops when end token is reached or max length is exceeded.
    """

    model.eval()

    current_char = torch.tensor([[char_to_index['<SOS>']]]).to(device)

    generated_name = []

    eos_idx = char_to_index['<EOS>']
    sos_idx = char_to_index['<SOS>']

    for step in range(max_length):
        with torch.no_grad():
            output = model(current_char)

        output = output[:, -1, :]

        # Remove SOS
        output[:, sos_idx] = -float('inf')

        # Temperature scaling
        output = output / temperature

        probs = torch.softmax(output, dim=-1).squeeze()

        # TOP-K filtering
        topk_probs, topk_indices = torch.topk(probs, top_k)

        # TOP-P (nucleus filtering)
        sorted_probs, sorted_indices = torch.sort(topk_probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=0)

        cutoff = cumulative_probs > top_p
        if torch.any(cutoff):
            sorted_probs = sorted_probs[:torch.where(cutoff)[0][0] + 1]
            sorted_indices = sorted_indices[:len(sorted_probs)]

        # Normalize
        sorted_probs = sorted_probs / sorted_probs.sum()

        # Sample
        next_idx = torch.multinomial(sorted_probs, 1).item()
        next_char_idx = topk_indices[sorted_indices[next_idx]].item()

        next_char = index_to_char[next_char_idx]

        # Dynamic EOS encouragement
        if step > 4 and torch.rand(1).item() < 0.3:
            break

        if next_char == '<EOS>':
            break

        # Repetition penalty
        if len(generated_name) >= 2 and next_char == generated_name[-1]:
            continue

        generated_name.append(next_char)

        current_char = torch.tensor([[next_char_idx]]).to(device)

    # Ensure minimum length
    if len(generated_name) < 3:
        return generate_name(model, char_to_index, index_to_char)

    return ''.join(generated_name)

### E. Training all the models

In [ ]:
# VANILLA RNN SETUP

"""
In this, we define hyperparameters for the Vanilla RNN model.
These include embedding size, hidden layer size, learning rate, and number of epochs.
Then, model is initialized using vocabulary size and chosen hyperparameters.
Further, we compute total number of trainable parameters to understand model complexity.
After that, loss function and optimizer are initialized for training.
Finally, model is trained on dataset.
"""

# Define hyperparameters
rnn_embedding_dim = 64
rnn_hidden_size = 128
rnn_learning_rate = 0.002
rnn_epochs = 10

# Initialize model
rnn_model = VanillaRNN(len(vocab), rnn_embedding_dim, rnn_hidden_size)

# Print model details
print("===== Vanilla RNN Model =====")
print("Embedding Dim:", rnn_embedding_dim)
print("Hidden Size:", rnn_hidden_size)
print("Learning Rate:", rnn_learning_rate)

# Count parameters
rnn_params = count_parameters(rnn_model)
print("Total Trainable Parameters:", rnn_params)

# Setup training
rnn_criterion, rnn_optimizer = setup_training(rnn_model, rnn_learning_rate)

# Train model
train_model(rnn_model, dataloader, rnn_criterion, rnn_optimizer, rnn_epochs)

# Generate sample names
print("\nGenerated Names (RNN):")
for _ in range(5):
    print(generate_name(rnn_model, char_to_index, index_to_char))

===== Vanilla RNN Model =====
Embedding Dim: 64
Hidden Size: 128
Learning Rate: 0.002
Total Trainable Parameters: 29850
Epoch [1/10], Loss: 2.4079
Epoch [2/10], Loss: 2.0375
Epoch [3/10], Loss: 1.9625
Epoch [4/10], Loss: 1.9318
Epoch [5/10], Loss: 1.9118
Epoch [6/10], Loss: 1.9041
Epoch [7/10], Loss: 1.9031
Epoch [8/10], Loss: 1.8826
Epoch [9/10], Loss: 1.8780
Epoch [10/10], Loss: 1.8876

Generated Names (RNN):
shanit
ananana
deshanan
kanananu
panunusha


In [ ]:
# BLSTM SETUP

"""
In this, we define hyperparameters for the BLSTM model.
Here, model is initialized with bidirectional LSTM layers.
Which process sequence in both forward and backward directions.
Also, we calculate total trainable parameters to compare with other models.
Then, training setup is initialized using loss function and optimizer.
Finally, model is trained on dataset.
"""

# Define hyperparameters
blstm_embedding_dim = 64
blstm_hidden_size = 128
blstm_learning_rate = 0.002
blstm_epochs = 10

# Initialize model
blstm_model = BLSTMModel(len(vocab), blstm_embedding_dim, blstm_hidden_size)

# Print model details
print("===== BLSTM Model =====")
print("Embedding Dim:", blstm_embedding_dim)
print("Hidden Size:", blstm_hidden_size)
print("Learning Rate:", blstm_learning_rate)
print("Bidirectional: True")

# Count parameters
blstm_params = count_parameters(blstm_model)
print("Total Trainable Parameters:", blstm_params)

# Setup training
blstm_criterion, blstm_optimizer = setup_training(blstm_model, blstm_learning_rate)

# Train model
train_model(blstm_model, dataloader, blstm_criterion, blstm_optimizer, blstm_epochs)

# Generate sample names
print("\nGenerated Names (BLSTM):")
for _ in range(5):
    print(generate_name(blstm_model, char_to_index, index_to_char))

===== BLSTM Model =====
Embedding Dim: 64
Hidden Size: 128
Learning Rate: 0.002
Bidirectional: True
Total Trainable Parameters: 602266
Epoch [1/10], Loss: 2.2788
Epoch [2/10], Loss: 1.0931
Epoch [3/10], Loss: 0.7624
Epoch [4/10], Loss: 0.7045
Epoch [5/10], Loss: 0.6821
Epoch [6/10], Loss: 0.6717
Epoch [7/10], Loss: 0.6656
Epoch [8/10], Loss: 0.6608
Epoch [9/10], Loss: 0.6568
Epoch [10/10], Loss: 0.6534

Generated Names (BLSTM):
poharohb
oohbnun
cvidikit
oochanoc
chan


In [ ]:
# RNN + ATTENTION SETUP

"""
In this, we define hyperparameters for the RNN with Attention model.
Also, model is initialized with an additional attention mechanism
Helps focus on important parts of the sequence.
Further, we compute the number of trainable parameters to understand the model size.
Then, training components are initialized and the model is trained.
Finally, model is trained on dataset.
"""

# Define hyperparameters
attn_embedding_dim = 64
attn_hidden_size = 128
attn_learning_rate = 0.002
attn_epochs = 10

# Initialize model
attn_model = RNNAttentionModel(len(vocab), attn_embedding_dim, attn_hidden_size)

# Print model details
print("===== RNN with Attention Model =====")
print("Embedding Dim:", attn_embedding_dim)
print("Hidden Size:", attn_hidden_size)
print("Learning Rate:", attn_learning_rate)
print("Attention: Dot-product")

# Count parameters
attn_params = count_parameters(attn_model)
print("Total Trainable Parameters:", attn_params)

# Setup training
attn_criterion, attn_optimizer = setup_training(attn_model, attn_learning_rate)

# Train model
train_model(attn_model, dataloader, attn_criterion, attn_optimizer, attn_epochs)

# Generate sample names
print("\nGenerated Names (RNN with Attention Model):")
for _ in range(5):
    print(generate_name(attn_model, char_to_index, index_to_char))

===== RNN with Attention Model =====
Embedding Dim: 64
Hidden Size: 128
Learning Rate: 0.002
Attention: Dot-product
Total Trainable Parameters: 46362
Epoch [1/10], Loss: 2.4945
Epoch [2/10], Loss: 2.0966
Epoch [3/10], Loss: 2.0195
Epoch [4/10], Loss: 1.9838
Epoch [5/10], Loss: 1.9679
Epoch [6/10], Loss: 1.9493
Epoch [7/10], Loss: 1.9547
Epoch [8/10], Loss: 1.9299
Epoch [9/10], Loss: 1.9307
Epoch [10/10], Loss: 1.9254

Generated Names (RNN with Attention Model):
anunananuyan
ditararu
arununita
anarushananu
anuna


In [ ]:
# MODEL SIZE CALCULATION

# Vanilla RNN
rnn_params, rnn_size = get_model_size(rnn_model)

print("Vanilla RNN Parameters:", rnn_params)
print("Vanilla RNN Model Size (MB):", round(rnn_size, 4))

# BLSTM Model
blstm_params, blstm_size = get_model_size(blstm_model)

print("\nBLSTM Parameters:", blstm_params)
print("BLSTM Model Size (MB):", round(blstm_size, 4))

# Attention Model
attn_params, attn_size = get_model_size(attn_model)

print("\nRNN + Attention Parameters:", attn_params)
print("RNN + Attention Model Size (MB):", round(attn_size, 4))

Vanilla RNN Parameters: 29850
Vanilla RNN Model Size (MB): 0.1139

BLSTM Parameters: 602266
BLSTM Model Size (MB): 2.2975

RNN + Attention Parameters: 46362
RNN + Attention Model Size (MB): 0.1769


## Task 2: Quantitative Evaluation

In [ ]:
def is_valid_name(name):
    """
    In this function, we checks whether generated name is valid or not.
    Here, check whether it satisfies basic length constraints.
    """

    return 2 <= len(name) <= 15

In [ ]:
def generate_multiple_names(model, char_to_index, index_to_char, num_names=100):
    """
    In this function, we generates multiple valid names using trained model.
    This return all the valid names for model.
    """

    generated_names = []

    while len(generated_names) < num_names:
        name = generate_name(model, char_to_index, index_to_char)

        # Keep only valid names
        if is_valid_name(name):
            generated_names.append(name)

    return generated_names

In [ ]:
def filter_realistic_names(names):
    """
    In this function, we filter names that look unrealistic.
    Here, names with excessive repeated characters are removed.
    Also, names with very low character variety are filtered out.
    So, overall quality of generated names is improved.
    """

    filtered = []

    for name in names:
        # Remove names with too many repeated chars
        if any(name.count(ch) > 3 for ch in name):
            continue

        # Remove very weird patterns
        if len(set(name)) < 3:
            continue

        filtered.append(name)

    return filtered

In [ ]:
# Converting training names into a set

def get_training_set(names):
    """
    In this function, we convert list of training names into a set.
    Because, set will allows faster lookup operations during evaluation.
    So, used when checking whether generated name exists in training data.
    """

    return set(names)

In [ ]:
# Create training set
training_set = get_training_set(names)

In [ ]:
# Checking if two names are similar

def is_similar(name1, name2, threshold=0.8):
    """
    In this function, we check similarity between two names using sequence matching.
    Also, similarity score is computed, and if it exceeds a threshold, names are considered similar.
    Helps in identifying names that are close to already known names.
    """

    return SequenceMatcher(None, name1, name2).ratio() > threshold

In [ ]:
# Computing novelty using similarity instead of exact match

def compute_novelty(generated_names, training_set):
    """
    In this function, we compute novelty based on similarity instead of exact matching.
    Here, each generated name is compared with subset of training names.
    So, if a generated name is too similar to any training name, it is not considered novel.
    Hence, this provides measure of novelty in names generated by models.
    """

    novel_count = 0

    training_list = list(training_set)

    for gen_name in generated_names:
        is_novel = True

        # Compare with some training names (not all for speed)
        for train_name in training_list[:500]:   # limit for efficiency
            if is_similar(gen_name, train_name):
                is_novel = False
                break

        if is_novel:
            novel_count += 1

    novelty_rate = novel_count / len(generated_names)

    return novelty_rate

In [ ]:
# Computing diversity of generated names

def compute_diversity(generated_names):
    """
    In this function, we calculate diversity of generated names.
    Here, we count unique names present out of total generated names.
    So, higher diversity means that model is producing different outputs.
    """

    unique_names = set(generated_names)

    diversity = len(unique_names) / len(generated_names)

    return diversity

In [ ]:
# Generating names and computing evaluation metrics

def evaluate_model(model, char_to_index, index_to_char, training_set, num_samples=200):
    """
    In this function, we evaluate model using generated names.
    Here, multiple names are generated using model.
    Then, invalid names are filtered out.
    Further, novelty and diversity metrics are computed.
    So, metrics help in comparing performance of different models.
    """

    # Generate names
    generated_names = generate_multiple_names(
        model, char_to_index, index_to_char, num_samples
    )
    generated_names = filter_realistic_names(generated_names)

    # Compute metrics
    novelty = compute_novelty(generated_names, training_set)
    diversity = compute_diversity(generated_names)

    return novelty, diversity, generated_names

In [ ]:
# Evaluating Vanilla RNN
rnn_novelty, rnn_diversity, rnn_generated = evaluate_model(
    rnn_model, char_to_index, index_to_char, training_set
)

# Evaluating BLSTM
blstm_novelty, blstm_diversity, blstm_generated = evaluate_model(
    blstm_model, char_to_index, index_to_char, training_set
)

# Evaluating RNN Attention Model
attn_novelty, attn_diversity, attn_generated = evaluate_model(
    attn_model, char_to_index, index_to_char, training_set
)

In [ ]:
print("===== Evaluation Results =====\n")

print("Vanilla RNN:")
print("Novelty Rate:", round(rnn_novelty, 3))
print("Diversity:", round(rnn_diversity, 3))

print("\nBLSTM:")
print("Novelty Rate:", round(blstm_novelty, 3))
print("Diversity:", round(blstm_diversity, 3))

print("\nRNN with Attention:")
print("Novelty Rate:", round(attn_novelty, 3))
print("Diversity:", round(attn_diversity, 3))

===== Evaluation Results =====

Vanilla RNN:
Novelty Rate: 0.736
Diversity: 0.591

BLSTM:
Novelty Rate: 0.969
Diversity: 0.979

RNN with Attention:
Novelty Rate: 0.681
Diversity: 0.748


In [ ]:
print("\n===== Sample Generated Names =====\n")

print("RNN Samples:")
print(rnn_generated[:10])

print("\nBLSTM Samples:")
print(blstm_generated[:10])

print("\nRNN with Attention Model Samples:")
print(attn_generated[:10])


===== Sample Generated Names =====

RNN Samples:
['ananit', 'kanan', 'shanan', 'shanitanu', 'panunun', 'kunan', 'kunitana', 'anushanun', 'anurur', 'ananu']

BLSTM Samples:
['zochar', 'ooubhroh', 'ochro', 'ochith', 'ppowocharv', 'dehikubh', 'dhavit', 'pavar', 'oochani', 'chbh']

RNN with Attention Model Samples:
['kanik', 'shanu', 'ditan', 'nuyarar', 'araranununu', 'anunura', 'shana', 'nunan', 'anika', 'shani']
